## Lorenz-63 and Chaos
  
- Lorenz-63 model is a system of ODEs that has chaotic solutions
  - see: https://docs.dart.ucar.edu/en/latest/guide/lorenz-63-model.html  or  https://en.wikipedia.org/wiki/Lorenz_system
- simplified mathematical representation of atmospheric convection
- applications in various fields, including laser physics, dynamos, and even electric circuits
- tiny changes in initial conditions yield completely different and unpredictable trajectories
- Edward Lorenz showed that the system switches between the two lobes chaotically
- Another visualisation: [Malkus waterwheel](https://en.wikipedia.org/wiki/Malkus_waterwheel)

In [ ]:
import numpy as np
from IPython.display import HTML, Image, display
from matplotlib import colors
from mpl_toolkits.mplot3d import Axes3D
import matplotlib.pyplot as plt
import matplotlib.animation as animation
## uncomment if ipympl is installed (a backend that helps to get better interactive 3d plots)
# %matplotlib ipympl

### Lorenz63 System

- Initial Condition: $\mathbf{z}_0=(x_0, y_0, z_0)^{\mathrm{T}}=\begin{pmatrix} -0.587 \\ -0.563 \\ 16.870 \end{pmatrix}$
- Vector Field (parameters $\sigma=10$, $\rho=28$, $\beta=8/3$):
    $$
    f(\mathbf{z})=
    \begin{pmatrix}
    \sigma(y-x) \\
    x(\rho-z)-y \\
    xy-\beta z
    \end{pmatrix}=
    \begin{pmatrix}
    10(y-x) \\
    x(28-z)-y \\
    xy-\frac{8}{3}z
    \end{pmatrix}.
    $$

### Task a: Implementation

In [ ]:
def lorenz(xyz, rho, sigma, beta):
    """
    Parameters
    xyz : array-like, shape (3,)
       Point of interest in three-dimensional space.
    sigma, rho, beta : float
       Parameters defining the Lorenz attractor.

    Returns:
    xyz_dot : array, shape (3,)
       Values of the Lorenz attractor's partial derivatives at *xyz*.
    """
    x, y, z = xyz # unpack
    x_dot = sigma * (y - x) 
    y_dot = x * (rho - z) - y 
    z_dot = x * y - beta * z 
    return np.array([x_dot, y_dot, z_dot])  # np.array to support arithmetic ops on array

#### Computation of Trajectory $\{\mathbf{z}_n\}$

In [ ]:
def calculate_z(z_0, rho, sigma, beta, dt_system, h, t_end):
    """
    Computes the trajectory of the Lorenz system using the discrete dynamical system formalism.
    
    Parameters:
    z_0 : array-like, shape (3,)
        Initial condition.
    rho, sigma, beta : float
        Lorenz parameters.
    dt_system : float
        System time step (Delta t), corresponds to the index step n -> n+1.
    h : float
        Internal computational step size for the Euler scheme.
    t_end : float
        End time of the simulation.
        
    Returns:
    data : array, shape (num_system_steps + 1, 3)
        The trajectory states z_n at times t = n * dt_system.
    time : array, shape (num_system_steps + 1,)
        The corresponding time points.
    """
    # Calculate number of system steps (indices n)
    num_system_steps = int(t_end / dt_system)
    
    # Calculate number of internal Euler steps per system step
    internal_steps = int(dt_system / h)
    
    # Pre-allocate output array (include initial condition)
    data = np.empty((num_system_steps + 1, 3))
    time = np.linspace(0, t_end, num_system_steps + 1)
    
    zt = np.array(z_0)
    data[0] = zt  # Store initial condition z_0
    
    # Loop over system steps n = 0, ..., N-1 to compute z_{n+1}
    for n in range(num_system_steps):
        # Perform internal Euler steps to approximate the evolution function Psi
        # This advances the state from t = n*dt_system to t = (n+1)*dt_system
        for k in range(internal_steps):
            zt = zt + h * lorenz(zt, rho, sigma, beta)
        
        # Store the state at the end of the system step
        data[n+1] = zt

    return data, time

### Question
- We integrate with $\Delta t$ but only save `zt` every `save_every_nstep` step. Why might we do this in real models?

In [ ]:
# System parameters (Formalism: Delta t = 0.1, h = 0.01)
dt_system = 0.1      # System time step (Delta t)
h = 0.01             # Internal computational step size
t_end = 10.0         # Simulation end time

# Set the initial conditions
z_0 = np.array([-0.587, -0.563, 16.870])

# Lorenz parameters
rho = 28.0
sigma = 10.0
beta = 8.0/3

# Compute trajectory
curve, time_points = calculate_z(z_0, rho, sigma, beta, dt_system, h, t_end)
## if needed, store the resulting reference trajectory in the csv file for later use
# np.savetxt('lorenz_data.csv', curve, delimiter=',')

### Task b:

In [ ]:
means = np.mean(curve, axis=0)
stds = np.std(curve, axis=0)

for label, mean, std in zip(['x', 'y', 'z'], means, stds):
    print(f"Mean {label}: {mean}, Std {label}: {std}")

### Task c: Plotting the Lorenz Attractor

**x-component**

In [ ]:
x = curve[:, 0]
y = curve[:, 1]
z = curve[:, 2]

fig = plt.figure(figsize=(9,6))
ax = fig.add_subplot()
ax.plot(time_points, x, label="x(t)")
ax.set_xlabel("t")
ax.set_ylabel("X")
ax.grid(True)
ax.legend()
ax.set_title(f"Lorenz Attractor | rho={rho}  sigma={sigma}  beta={beta:.3f}")
plt.show()

### Task d: 3D-Plot

In [ ]:
fig = plt.figure(figsize=(9,9))
ax = fig.add_subplot(projection='3d')
# modify ax.get_proj method: scale the x and y by 0.9 and 0.7 of the projection matrix
ax.get_proj = lambda: Axes3D.get_proj(ax) @ np.diag([0.9, 0.7, 1, 1])  # 
ax.plot(x, y, z, lw=0.9, c='k')
ax.set_xlabel("X")
ax.set_ylabel("Y")
ax.set_zlabel("Z")
ax.set_title(f"Lorenz Attractor | rho={rho}  sigma={sigma}  beta={beta:.3f}")
plt.show()

### Task e: Second Trajectory and Plotting (+Animation)

In [ ]:
change = 0.01
z_0_changed = z_0 + change
curve2, time_points = calculate_z(z_0_changed, rho, sigma, beta,
                                  dt_system, h, t_end)
x2 = curve2[:, 0]
y2 = curve2[:, 1]
z2 = curve2[:, 2]

# Create figure
fig, ax = plt.subplots(1, 1, subplot_kw={'projection': '3d'}, figsize=(12, 6))

# Initialize plots
line1, = ax.plot([], [], [], lw=1, c='red', label=f"z0+{change}")
point1, = ax.plot([], [], [], 'ro', markersize=5)
point2, = ax.plot([], [], [], 'ko', markersize=5)

ax.plot(x, y, z, lw=0.5, c='gray', ls='dashed', label=f"z0")
ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_zlabel('Z')

ax.set_title(f"rho={rho}  sigma={sigma}  beta={beta:.3f}")
plt.legend()

# Initialize animation function
def init():
    line1.set_data([], [])
    line1.set_3d_properties([])
    point1.set_data([], [])
    point1.set_3d_properties([])
    point2.set_data([], [])
    point2.set_3d_properties([])
    return line1, point1

# Update function
def update(i):
    line1.set_data(x2[:i], y2[:i])
    line1.set_3d_properties(z2[:i])
    point1.set_data([x2[i]], [y2[i]])
    point1.set_3d_properties([z2[i]])
    point2.set_data([x[i]], [y[i]])
    point2.set_3d_properties([z[i]])
    return line1, point1

# Create the animation
ani = animation.FuncAnimation(fig, update, frames=len(z), init_func=init,
                              interval=100, blit=True)

plt.close()  # finish/close animation

## if ffmpeg installed
## Display the animation
# HTML(ani.to_html5_video())

ani.save(filename="lorenz63.gif", writer="pillow")
display(Image(filename="lorenz63.gif"))

### Task f: RMSE of Trajectories
 $$\text{RMSE} = \sqrt{\frac{1}{N}\sum_{n=1}^N \| \mathbf{z}^\ast_n - \mathbf{z}_n \|_2^2 } = \sqrt{\frac{1}{N}\sum_{n=1}^N \left( (\mathbf{z}^*_n - \mathbf{z}_n)^T (\mathbf{z}^*_n - \mathbf{z}_n) \right)} $$

In [ ]:
change = 0.01
z_0_changed = z_0 + change
curve2, time_points = calculate_z(z_0_changed, rho, sigma, beta,
                                  dt_system, h, t_end)
rmse = np.sqrt( np.mean( (curve2 - curve)**2 ) )
print(f"RMSE = {rmse}")

### Task e: RMSE over Time

In [ ]:
# System parameters (Formalism: Delta t = 0.1, h = 0.01)
dt_system = 0.1      # System time step (Delta t), corresponds to index n -> n+1
h = 0.01             # Internal computational step size for Euler scheme
t_end = 40           # Simulation end time (increased from 10 to 40)

# Calculate number of system steps (indices n)
num_system_steps = int(t_end / dt_system)

# Set the initial conditions
z_0 = np.array([-0.587, -0.563, 16.870])

# Lorenz parameters
rho = 28.0
sigma = 10.0
beta = 8.0/3

# Generate time points for system indices (t_n = n * dt_system)
time_points = np.linspace(0, t_end, num_system_steps + 1)

# Calculate trajectories using the updated formalism
curve, _ = calculate_z(z_0, rho, sigma, beta, dt_system, h, t_end)
curve_perturbed, _ = calculate_z(z_0 + 0.01 * np.array([1, 1, 1]), rho, sigma, beta, dt_system, h, t_end)

# Calculate RMSE over time (cumulative)
rmse_values = []
rmse_time_points = []

for i in range(1, len(curve)):
    # Compute RMSE up to step i (Euclidean norm across all 3 components)
    diff = curve_perturbed[:i] - curve[:i]
    # RMSE = sqrt(1/i * sum_{n=1}^{i} ||z*_n - z_n||^2)
    rmse = np.sqrt(np.mean(np.sum(diff**2, axis=1)))
    rmse_values.append(rmse)
    rmse_time_points.append(time_points[i])

# Plot RMSE vs time
plt.figure(figsize=(10, 6))
plt.plot(rmse_time_points, rmse_values, 'b-', linewidth=2)
plt.xlabel('Time t')
plt.ylabel('RMSE')
plt.title('RMSE Between Two Chaotic Trajectories Over Time (t ∈ [0, 40])')
plt.grid(True)
plt.yscale('log')  # Log scale to better visualize exponential growth
plt.show()

# Print final RMSE
print(f"Final RMSE at t={t_end}: {rmse_values[-1]:.6f}")